# Spotify charts 

In [1]:
import requests
url = "https://charts.spotify.com/charts/view/citytoptrack-vancouver-weekly/2025-05-01"

res = requests.get(url)

In [2]:
res

<Response [200]>

In [4]:
#!/usr/bin/env python3
"""
Pull a Spotify chart to CSV.

Setup: create tokens.json with the refresh_token from the /api/token response:
    {"refresh_token": "AQD..."}

Usage:
    python charts.py citytoptrack-vancouver-weekly
    python charts.py regional-ca-weekly 2026-07-17 out.csv
"""
import csv
import json
import sys
from pathlib import Path

import requests

CLIENT_ID = "44407c71b3b24071865aaa4fea948a15"
TOKENS = Path("tokens.json")


def access_token():
    rt = json.loads(TOKENS.read_text())["refresh_token"]
    r = requests.post("https://accounts.spotify.com/api/token", data={
        "grant_type": "refresh_token",
        "refresh_token": rt,
        "client_id": CLIENT_ID,
    }, timeout=30)
    r.raise_for_status()
    d = r.json()
    # Spotify rotates refresh tokens on PKCE clients - persist or this works once.
    if d.get("refresh_token"):
        TOKENS.write_text(json.dumps({"refresh_token": d["refresh_token"]}))
    return d["access_token"]


def fetch(alias, date="latest"):
    r = requests.get(
        f"https://charts-spotify-com-service.spotify.com/auth/v0/charts/{alias}/{date}",
        headers={
            "authorization": f"Bearer {access_token()}",
            "app-platform": "Browser",
            "spotify-app-version": "0.0.0.production",
            "accept": "application/json",
        }, timeout=30)
    r.raise_for_status()
    return r.json()


def rows(payload):
    date = payload["displayChart"]["date"]
    for e in payload["entries"]:
        c, t = e["chartEntryData"], e["trackMetadata"]
        yield {
            "date": date,
            "rank": c["currentRank"],
            "previous_rank": c.get("previousRank"),
            "peak_rank": c.get("peakRank"),
            "peak_date": c.get("peakDate"),
            "weeks_on_chart": c.get("appearancesOnChart"),
            "consecutive_weeks": c.get("consecutiveAppearancesOnChart"),
            "entry_rank": c.get("entryRank"),
            "entry_date": c.get("entryDate"),
            "status": c.get("entryStatus"),
            "track": t["trackName"],
            "artists": ", ".join(a["name"] for a in t.get("artists", [])),
            "label": ", ".join(l["name"] for l in t.get("labels", [])),
            "release_date": t.get("releaseDate"),
            "track_id": t["trackUri"].rsplit(":", 1)[-1],
        }


def main():
    alias = "citytoptrack-vancouver-weekly"
    date = "latest"
    out = f"{alias}-{date}.csv"

    data = list(rows(fetch(alias, date)))
    with open(out, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(data[0]))
        w.writeheader()
        w.writerows(data)
    print(f"{len(data)} rows -> {out}")

In [5]:
main()

100 rows -> citytoptrack-vancouver-weekly-latest.csv


# Wikipedia Analytics API

## Daily Page views for different articles

In [6]:
# Example 1 — Daily pageviews for two articles, en.wikipedia, all of 2025
# Endpoint: /metrics/pageviews/per-article/{project}/{access}/{agent}/{article}/{granularity}/{start}/{end}

import requests
import pandas as pd
import matplotlib.pyplot as plt

# Wikimedia asks that you identify yourself. Put a real contact here.
HEADERS = {"User-Agent": "zeitgeist-test (you@example.com)"}

BASE = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
PROJECT = "en.wikipedia.org"
ARTICLES = ["Bitcoin", "Artificial_intelligence"]
START, END = "20250101", "20251231"

# agent=user strips crawlers/bots. Without it the series is badly polluted.
AGENT = "user"
ACCESS = "all-access"  # try "desktop" / "mobile-web" later to see the split

frames = []
for article in ARTICLES:
    url = f"{BASE}/{PROJECT}/{ACCESS}/{AGENT}/{article}/daily/{START}/{END}"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json()["items"])
    frames.append(df[["timestamp", "article", "views"]])

daily = pd.concat(frames, ignore_index=True)
daily["date"] = pd.to_datetime(daily["timestamp"], format="%Y%m%d%H")

# One column per article, indexed by date
wide = daily.pivot(index="date", columns="article", values="views")

print(wide.head())
print()
print(wide.describe().round(0))

# ---- Raw daily ----
wide.plot(figsize=(13, 4), linewidth=0.8, title="Daily pageviews (raw), 2025")
plt.ylabel("views")
plt.show()

# ---- 7-day rolling mean ----
# The AI "slow climb" is invisible under daily noise; the smoothed line is
# where you'd actually see drift vs. spike.
wide.rolling(7).mean().plot(
    figsize=(13, 4), linewidth=1.4, title="7-day rolling mean, 2025"
)
plt.ylabel("views")
plt.show()

# ---- Normalized, so the two are comparable despite different scales ----
(wide / wide.mean()).rolling(7).mean().plot(
    figsize=(13, 4), linewidth=1.4, title="Normalized to each article's own 2025 mean"
)
plt.ylabel("views / mean")
plt.axhline(1.0, color="grey", linewidth=0.6, linestyle="--")
plt.show()

# ---- Biggest single-day spikes: sanity-check against real events ----
for article in wide.columns:
    print(f"\nTop 5 days — {article}")
    print(wide[article].nlargest(5))

ModuleNotFoundError: No module named 'pandas'